In [5]:
import pandas as pd
import re

In [6]:
# ========================== LOAD DATA ==========================
df = pd.read_csv('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/Cleaned_Verdict_Text.csv')

In [7]:
# ========================== LABELING FUNCTION ==========================
def get_decision_label(text):
    if not isinstance(text, str):
        return "Unknown"
    
    text_lower = text.lower()
    text_clean = re.sub(r'\s+', ' ', text)  # clean extra spaces
    
    # Check for final decision section
    if re.search(r'Վ\s*Ճ\s*Ռ\s*Ե\s*Ց|Վճռում է|Վճիռ', text):
        
        # Fully Approved
        if re.search(r'Հայցը բավարարել', text):
            # Check if partial
            if re.search(r'մասնակի|մասով|կարճել|հրաժարվել', text):
                return "Partially Approved"
            return "Fully Approved"
        
        # Rejected
        if re.search(r'Հայցը մերժել', text):
            return "Rejected"
        
        # Divorce cases
        if re.search(r'ամուսնությունը լուծել|ամուսնալուծել', text) and re.search(r'բավարարել', text):
            return "Approved (Divorce)"
        
        # Contract termination + compensation
        if re.search(r'պայմանագիրը լուծել|բռնագանձել', text):
            return "Fully Approved"
    
    # Partial withdrawal or case closed
    if re.search(r'հրաժարվել.*պահանջ|վարույթը կարճել', text_lower):
        return "Partially Approved / Closed"
    
    # Default fallback
    if "բավարարել" in text_lower:
        return "Approved"
    if "մերժել" in text_lower:
        return "Rejected"
    
    return "Unknown / Needs Manual Check"


In [8]:
# ========================== APPLY LABEL ==========================
df['Decision_Label'] = df['Verdict_Text'].apply(get_decision_label)

# ========================== RESULTS ==========================
print("Label Distribution:")
print(df['Decision_Label'].value_counts())

# Show full table
print("\n=== LABELED RESULTS ===")
selected_cols = [c for c in ['Case_Number', 'Parties', 'Claim_Type', 'Decision_Label'] if c in df.columns]
print(df[selected_cols])

# Save to new CSV
df.to_csv('/Users/tsovinarbabakhanyan/Desktop/Armenian Chat part/src/data/legal_analysis_labeled.csv', index=False, encoding='utf-8')
print("\n✅ Saved as 'legal_analysis_labeled.csv'")

Label Distribution:
Decision_Label
Partially Approved / Closed     1109
Approved                         847
Rejected                         110
Unknown / Needs Manual Check       7
Name: count, dtype: int64

=== LABELED RESULTS ===
          Case_Number               Decision_Label
0      ԵԴ2/3094/02/24  Partially Approved / Closed
1      ԵԴ2/8206/02/24  Partially Approved / Closed
2      ԵԴ2/9445/02/24  Partially Approved / Closed
3      ԵԴ2/9445/02/24  Partially Approved / Closed
4     ԵԴ2/11140/02/24  Partially Approved / Closed
...               ...                          ...
2068   ԵԴ2/4067/02/25                     Approved
2069   ԵԴ2/7722/02/25  Partially Approved / Closed
2070    ԼԴ/1337/02/25                     Approved
2071   ԵԴ2/8339/02/24                     Rejected
2072   ԵԴ2/8323/02/24                     Approved

[2073 rows x 2 columns]

✅ Saved as 'legal_analysis_labeled.csv'
